#RL Actor Critic PPO Policy/Agent for Motion Planning end to end in MetaDrive Simulator/Environment

You will:
- Formulate autonomous driving as an MDP
- Train a PPO agent (Stable-Baselines version)
- Evaluate generalization
- Deploy and visualize the policy
- Launch a Gradio demo


In [ ]:
!pip install -q git+https://github.com/metadriverse/metadrive.git
!pip install -q stable-baselines3 gymnasium torch gradio imageio

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 6.9 MB/s eta 0:00:00


In [ ]:
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'

In [ ]:
import numpy as np
import torch
import imageio
import gradio as gr

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

from metadrive import MetaDriveEnv
from metadrive.engine.engine_utils import close_engine

close_engine()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

In [ ]:
ENV_CONFIG = {
    "use_render": False,
    "num_scenarios": 50,
    "start_seed": 0,
    "traffic_density": 0.1,
    "random_lane_width": True,
    "random_agent_model": True,
    "num_agents": 1,
    "horizon": 1000,

    # 🔴 CRITICAL: freeze sensors
    "sensors": {
        "lidar": (),
        "side_detector": (),
        "lane_line_detector": ()
    }
}


## 1. MetaDrive Environment (Single Agent, Safe Setup)

In [ ]:
close_engine()

env = MetaDriveEnv(ENV_CONFIG)
env = Monitor(env)

print('Observation space:', env.observation_space)
print('Action space:', env.action_space)

[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000


Observation space: Box(-0.0, 1.0, (261,), float32)
Action space: Box(-1.0, 1.0, (2,), float32)


## 2. PPO Actor–Critic Configuration

In [ ]:
policy_kwargs = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
    activation_fn=torch.nn.ReLU
)

model = PPO(
    'MlpPolicy',
    env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs=policy_kwargs,
    verbose=1,
)

Using cuda device
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3. Train PPO Agent

In [ ]:
model.learn(total_timesteps=100_000)
model.save('ppo_metadrive_safe')
print('Model saved')

[WARNING] Assets folder doesn't exist. Begin to download assets... (base_engine.py:773)
[INFO] Pull assets from https://github.com/metadriverse/metadrive/releases/download/MetaDrive-0.4.3/assets.zip to /usr/local/lib/python3.12/dist-packages/metadrive/assets.zip

[INFO] Extracting assets.
[INFO] Successfully download assets, version: 0.4.3. MetaDrive version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 0, Num Scenarios : 50


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 560      |
|    ep_rew_mean     | 4        |
| time/              |          |
|    fps             | 188      |
|    iterations      | 1        |
|    time_elapsed    | 10       |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 677         |
|    ep_rew_mean          | 5.45        |
| time/                   |             |
|    fps                  | 160         |
|    iterations           | 2           |
|    time_elapsed         | 25          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.014575329 |
|    clip_fraction        | 0.223       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.83       |
|    explained_variance   | 0.0122      |
|    learning_rate        | 0.

## 4. Evaluation on Unseen Maps

In [ ]:
close_engine()

TEST_ENV_CONFIG = ENV_CONFIG.copy()
TEST_ENV_CONFIG["start_seed"] = 1000  # unseen scenarios

test_env = MetaDriveEnv(TEST_ENV_CONFIG)


def evaluate(model, env, episodes=5):
    rewards = []
    for ep in range(episodes):
        obs, _ = env.reset()
        done = False
        ep_r = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, _ = env.step(action)
            done = term or trunc
            ep_r += r
        rewards.append(ep_r)
    return np.mean(rewards), np.std(rewards)

print('Eval reward:', evaluate(model, test_env))

[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 1000, Num Scenarios : 50


Eval reward: (np.float64(87.24649979920248), np.float64(30.789511660664523))


## 5. Policy Deployment & Visualization

In [ ]:
close_engine()

render_env = MetaDriveEnv(ENV_CONFIG)

obs, _ = render_env.reset()
frames = []

for _ in range(800):
    action, _ = model.predict(obs, deterministic=True)
    obs, r, term, trunc, _ = render_env.step(action)
    frame = render_env.render(mode='top_down', screen_size=(500, 500))
    frames.append(frame)
    if term or trunc:
        break

render_env.close()
imageio.mimsave('ppo_safe_demo.gif', frames, fps=10)
print('Saved ppo_safe_demo.gif')

[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 0, Num Scenarios : 50
/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/li

Saved ppo_safe_demo.gif


In [ ]:
from metadrive import MetaDriveEnv
from metadrive.engine.engine_utils import close_engine
import imageio
import os

def generate_gif(seed, traffic_density, filename):
    close_engine()

    env = MetaDriveEnv({
        **ENV_CONFIG,
        "start_seed": seed,
        "traffic_density": traffic_density,
    })

    obs, _ = env.reset()
    frames = []

    for _ in range(600):
        action, _ = model.predict(obs, deterministic=True)
        obs, r, term, trunc, _ = env.step(action)

        frame = env.render(
            mode="top_down",
            screen_size=(500, 500)
        )
        frames.append(frame)

        if term or trunc:
            break

    env.close()
    imageio.mimsave(filename, frames, fps=10)
    return filename


In [ ]:
os.makedirs("results", exist_ok=True)

GIF_INDEX = {}

for seed in [1008, 1010, 1012]:
    for density in [0.05, 0.1, 0.2]:
        name = f"results/seed{seed}_density{density}.gif"
        generate_gif(seed, density, name)
        GIF_INDEX[(seed, density)] = name

print("GIFs generated:", len(GIF_INDEX))


[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 1008, Num Scenarios : 50
[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: glxGraphicsPipe
[INFO] Start Scenario Index: 1008, Num Scenarios : 50
[INFO] Environment: MetaDriveEnv
[INFO] MetaDrive version: 0.4.3
[INFO] Sensors: [lidar: Lidar(), side_detector: SideDetector(), lane_line_detector: LaneLineDetector()]
[INFO] Render Mode: none
[INFO] Horizon (Max steps per agent): 1000
[INFO] Assets version: 0.4.3
[INFO] Known Pipes: gl

GIFs generated: 9


## 6. Gradio Interactive Demo

In [ ]:
import gradio as gr

def show_gif(seed, traffic_density):
    key = (int(seed), float(traffic_density))
    return GIF_INDEX.get(key)


In [ ]:

demo = gr.Interface(
    fn=show_gif,
    inputs=[
        gr.Dropdown(
            choices=[1008, 1010, 1012],
            label="Scenario Seed",
            value=1000
        ),
        gr.Dropdown(
            choices=[0.05, 0.1, 0.2],
            label="Traffic Density",
            value=0.1
        ),
    ],
    outputs=gr.Image(type="filepath"),
    title="PPO MetaDrive Results Viewer",
    description="""
    This demo displays pre-generated PPO rollouts.
    MetaDrive runs offline; Gradio only visualizes results.
    """
)

demo.launch(share=True, debug=True)


/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: 1000 or set allow_custom_value=True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/components/clear_button.py:105: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  self.click(
/usr/local/lib/python3.12/dist-packages/gradio/interface.py:850: DeprecationWarning: The 'show_api' parameter in event listeners will be removed in Gradio 6.0. You will need to use the 'api_visibility' parameter instead. To replicate show_api=False, in Gradio 6.0, use api_visibility='undocumented'.
  _clear_btn.add(self.input_components + self.output_components)
/usr/local/lib/python3.12/dist-packages/jupyter_clie

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


* Running on public URL: https://e8b9df391d6ffab4e4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1341: DeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
